In [4]:
import numpy as np
import random

# --- FIXED ENVIRONMENT SETUP ---
# Changing this to match Test Case 2 (Obstacle in the center)
MIN_BOUND, MAX_BOUND = 0.0, 20.0
START = np.array([2.0, 2.0])
TARGET = np.array([18.0, 18.0])
OBSTACLES = [(10.0, 10.0, 4.0)]  # Center X, Center Y, Radius
NUM_WAYPOINTS = 3                 # Fixed complexity

def calculate_cost(wolf_position):
    waypoints = wolf_position.reshape(-1, 2)
    full_path = np.vstack([START, waypoints, TARGET])

    # 1. Total Distance
    distance = 0.0
    for i in range(len(full_path) - 1):
        distance += np.linalg.norm(full_path[i+1] - full_path[i])

    # 2. Obstacle Penalty
    penalty = 0.0
    for pt in waypoints:
        for obs_x, obs_y, obs_r in OBSTACLES:
            dist_to_obs = np.linalg.norm(pt - np.array([obs_x, obs_y]))
            if dist_to_obs <= obs_r:
                penalty += 1000.0 * (obs_r - dist_to_obs + 1)

    return distance + penalty

def grey_wolf_optimizer(num_wolves, max_iter):
    dimensions = NUM_WAYPOINTS * 2
    positions = np.random.uniform(MIN_BOUND, MAX_BOUND, (num_wolves, dimensions))

    alpha_pos, alpha_score = np.zeros(dimensions), float("inf")
    beta_pos, beta_score = np.zeros(dimensions), float("inf")
    delta_pos, delta_score = np.zeros(dimensions), float("inf")

    for iteration in range(max_iter):
        for i in range(num_wolves):
            positions[i] = np.clip(positions[i], MIN_BOUND, MAX_BOUND)
            fitness = calculate_cost(positions[i])

            if fitness < alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = fitness, positions[i].copy()
            elif fitness < beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = fitness, positions[i].copy()
            elif fitness < delta_score:
                delta_score, delta_pos = fitness, positions[i].copy()

        a = 2 - iteration * (2 / max_iter)

        for i in range(num_wolves):
            for j in range(dimensions):
                # Alpha update
                A1 = 2 * a * random.random() - a
                X1 = alpha_pos[j] - A1 * abs(2 * random.random() * alpha_pos[j] - positions[i, j])

                # Beta update
                A2 = 2 * a * random.random() - a
                X2 = beta_pos[j] - A2 * abs(2 * random.random() * beta_pos[j] - positions[i, j])

                # Delta update
                A3 = 2 * a * random.random() - a
                X3 = delta_pos[j] - A3 * abs(2 * random.random() * delta_pos[j] - positions[i, j])

                positions[i, j] = (X1 + X2 + X3) / 3

    best_waypoints = alpha_pos.reshape(-1, 2)
    return np.vstack([START, best_waypoints, TARGET]), alpha_score

# --- ONLY 2 INPUTS REQUIRED FROM USER ---
if __name__ == "__main__":
    print("=== Simplified GWO Path Planner ===")
    user_wolves = int(input("Enter number of Wolves (e.g., 30): "))
    user_iters = int(input("Enter number of Iterations (e.g., 100): "))

    print("\nRunning GWO optimization...")
    best_path, final_cost = grey_wolf_optimizer(user_wolves, user_iters)

    print("\n=== OPTIMAL PATH FOUND ===")
    print(f"Final Path Cost: {final_cost:.2f}")
    for idx, pt in enumerate(best_path):
        if idx == 0:
            print(f"  Start  -> ({pt[0]:.2f}, {pt[1]:.2f})")
        elif idx == len(best_path) - 1:
            print(f"  Target -> ({pt[0]:.2f}, {pt[1]:.2f})")
        else:
            print(f"  WP #{idx}  -> ({pt[0]:.2f}, {pt[1]:.2f})")


=== Simplified GWO Path Planner ===
Enter number of Wolves (e.g., 30): 10
Enter number of Iterations (e.g., 100): 20

Running GWO optimization...

=== OPTIMAL PATH FOUND ===
Final Path Cost: 25.92
  Start  -> (2.00, 2.00)
  WP #1  -> (2.33, 0.04)
  WP #2  -> (4.61, 3.76)
  WP #3  -> (15.18, 15.60)
  Target -> (18.00, 18.00)
